In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import gc

DATA_RAW = Path("../data/raw")
DATA_PROC = Path("../data/processed")
META_PATH = DATA_RAW / "meta_Beauty_and_Personal_Care.jsonl"

print(f"meta 文件: {META_PATH.stat().st_size / 1e9:.2f} GB")

# 先加载训练集需要的 item id 列表 (只关心这些商品)
item_features_old = pd.read_csv(DATA_PROC / "item_popularity.csv")
target_asins = set(item_features_old['parent_asin'].values)
print(f"训练集涉及的商品数: {len(target_asins):,}")

meta 文件: 2.84 GB
训练集涉及的商品数: 207,385


In [2]:
%%time
# 流式读取 1.03M 商品,只保留我们需要的 ~20.7 万个
# 内存峰值预计 < 500 MB (而不是 5-10 GB)

records = []
n_total = 0
n_kept = 0

with open(META_PATH, 'r') as f:
    for line in f:
        n_total += 1
        d = json.loads(line)
        asin = d.get('parent_asin')
        
        # 只保留训练集涉及的商品
        if asin not in target_asins:
            continue
        
        # 提取我们需要的 6 个字段
        record = {
            'parent_asin': asin,
            'price': d.get('price'),  # float or None
            'title': d.get('title', '') or '',
            'store': d.get('store', 'unknown') or 'unknown',
            'categories': d.get('categories', []) or [],
            'average_rating': d.get('average_rating'),
        }
        records.append(record)
        n_kept += 1
        
        if n_total % 200000 == 0:
            print(f"   已扫描 {n_total:,} 条, 命中 {n_kept:,}")

print(f"\n✅ 扫描完成: 总 {n_total:,} 条, 命中 {n_kept:,} 条")
print(f"   命中率: {n_kept/n_total*100:.1f}%")
print(f"   覆盖训练集: {n_kept/len(target_asins)*100:.1f}%")

# 转 DataFrame
df_meta = pd.DataFrame(records)
print(f"\n   df_meta shape: {df_meta.shape}")
print(f"   内存: {df_meta.memory_usage(deep=True).sum()/1e6:.1f} MB")
del records
gc.collect()


✅ 扫描完成: 总 1,028,914 条, 命中 207,385 条
   命中率: 20.2%
   覆盖训练集: 100.0%

   df_meta shape: (207385, 6)
   内存: 63.3 MB
CPU times: user 8.07 s, sys: 599 ms, total: 8.67 s
Wall time: 8.76 s


0

In [4]:
%%time

# ============================================
# 修复版: price 强制转 numeric, 非法值变 NaN
# ============================================

# 1. price 处理: 强制转 numeric (字符串/异常值 → NaN), 再加缺失指示器
print("=" * 60)
print("price 类型检查")
print("=" * 60)
print(f"price dtype (原): {df_meta['price'].dtype}")
print(f"price 样本类型分布:")
print(df_meta['price'].apply(type).value_counts().head())

# 关键: pd.to_numeric(errors='coerce') 把任何不能转的值变 NaN
df_meta['price'] = pd.to_numeric(df_meta['price'], errors='coerce')
print(f"\nprice dtype (转后): {df_meta['price'].dtype}")

# 加缺失指示器
df_meta['price_missing'] = df_meta['price'].isna().astype(int)
price_median = df_meta['price'].median()
df_meta['price'] = df_meta['price'].fillna(price_median)
print(f"\n✅ price 中位数: ${price_median:.2f}")
print(f"   price 缺失率 (含异常字符串): {df_meta['price_missing'].mean()*100:.1f}%")

# 看 price 分布是否合理
print(f"\nprice 描述统计:")
print(df_meta['price'].describe())

# 2. title 长度
df_meta['title_length'] = df_meta['title'].str.len()
print(f"\n" + "=" * 60)
print(f"title 长度分布:")
print(df_meta['title_length'].describe())

# 3. categories 层数
df_meta['n_categories'] = df_meta['categories'].apply(len)
print(f"\n" + "=" * 60)
print(f"categories 层数分布:")
print(df_meta['n_categories'].value_counts().sort_index())

# 4. 二级类目 (categories[1])
def get_sub_category(cats):
    if len(cats) >= 2:
        return cats[1]
    elif len(cats) == 1:
        return cats[0]
    else:
        return 'Unknown'

df_meta['sub_category_raw'] = df_meta['categories'].apply(get_sub_category)
print(f"\n" + "=" * 60)
print(f"二级类目 top 10:")
print(df_meta['sub_category_raw'].value_counts().head(10))

# Top-30 sub_category + Others
top_30_cats = df_meta['sub_category_raw'].value_counts().head(30).index.tolist()
df_meta['sub_category'] = df_meta['sub_category_raw'].apply(
    lambda x: x if x in top_30_cats else 'Others'
)
print(f"\n压缩后类目数: {df_meta['sub_category'].nunique()} (top 30 + Others)")
print(f"Others 占比: {(df_meta['sub_category']=='Others').mean()*100:.1f}%")

# 5. brand (store) - Top-100 + Others
top_100_brands = df_meta['store'].value_counts().head(100).index.tolist()
df_meta['brand'] = df_meta['store'].apply(
    lambda x: x if x in top_100_brands else 'Others'
)
print(f"\n" + "=" * 60)
print(f"压缩后品牌数: {df_meta['brand'].nunique()} (top 100 + Others)")
print(f"Others 占比: {(df_meta['brand']=='Others').mean()*100:.1f}%")

price 类型检查
price dtype (原): object
price 样本类型分布:
price
<class 'float'>       120697
<class 'NoneType'>     86687
<class 'str'>              1
Name: count, dtype: int64

price dtype (转后): float64

✅ price 中位数: $15.99
   price 缺失率 (含异常字符串): 41.8%

price 描述统计:
count    207385.000000
mean         21.169913
std          31.082751
min           0.010000
25%          13.990000
50%          15.990000
75%          18.480000
max        3398.000000
Name: price, dtype: float64

title 长度分布:
count    207385.000000
mean        116.832712
std          55.664737
min           0.000000
25%          65.000000
50%         115.000000
75%         168.000000
max         548.000000
Name: title_length, dtype: float64

categories 层数分布:
n_categories
2      2327
3      9345
4    116082
5     75166
6      4465
Name: count, dtype: int64

二级类目 top 10:
sub_category_raw
Hair Care                 55413
Skin Care                 45279
Foot, Hand & Nail Care    28143
Makeup                    23744
Tools & Accessories   

In [5]:
%%time

# ============================================
# 修订版 Cell 4: 增加 brand 到 top-500
# ============================================

# Top-500 brands (从 5 到 500)
top_500_brands = df_meta['store'].value_counts().head(500).index.tolist()
df_meta['brand'] = df_meta['store'].apply(
    lambda x: x if x in top_500_brands else 'Others'
)
print(f"压缩后品牌数: {df_meta['brand'].nunique()} (top 500 + Others)")
print(f"Others 占比 (top-500): {(df_meta['brand']=='Others').mean()*100:.1f}%")

# Label Encoding
sub_cat_to_id = {cat: i for i, cat in enumerate(df_meta['sub_category'].unique())}
df_meta['sub_category_id'] = df_meta['sub_category'].map(sub_cat_to_id)

brand_to_id = {brand: i for i, brand in enumerate(df_meta['brand'].unique())}
df_meta['brand_id'] = df_meta['brand'].map(brand_to_id)

print(f"\nsub_category_id 范围: 0 - {df_meta['sub_category_id'].max()}")
print(f"brand_id 范围: 0 - {df_meta['brand_id'].max()}")

# 最终特征列
meta_features_cols = [
    'parent_asin',
    'price',
    'price_missing',
    'title_length',
    'n_categories',
    'sub_category_id',
    'brand_id',
]
df_item_meta = df_meta[meta_features_cols].copy()
print(f"\n✅ 最终 meta 特征: {df_item_meta.shape}")
print(df_item_meta.head())
print(f"\n内存: {df_item_meta.memory_usage(deep=True).sum()/1e6:.1f} MB")

# 描述统计 — 检查所有数值列分布是否正常
print(f"\n数值统计:")
print(df_item_meta.describe())

压缩后品牌数: 501 (top 500 + Others)
Others 占比 (top-500): 65.0%

sub_category_id 范围: 0 - 30
brand_id 范围: 0 - 500

✅ 最终 meta 特征: (207385, 7)
  parent_asin  price  price_missing  title_length  n_categories  \
0  B0BWJGQ32Y  45.65              0           189             4   
1  B00N4LMZZK  14.99              0            69             5   
2  B01DX1OEFO   2.49              0            64             4   
3  B0B6PN78BY  59.99              0           161             4   
4  B01KQNJCNG  20.89              0            44             4   

   sub_category_id  brand_id  
0                0         0  
1                1         0  
2                2         1  
3                0         0  
4                1         0  

内存: 13.7 MB

数值统计:
               price  price_missing   title_length   n_categories  \
count  207385.000000  207385.000000  207385.000000  207385.000000   
mean       21.169913       0.418005     116.832712       4.338004   
std        31.082751       0.493232      55.664737

In [9]:
# brand_id=0 的数量
print(f"brand_id=0 的商品数: {(df_item_meta['brand_id']==0).sum():,}")
print(f"占比: {(df_item_meta['brand_id']==0).mean()*100:.1f}%")

# brand_id 前 5 分布
print(f"\nbrand_id 前 5 的分布:")
print(df_item_meta['brand_id'].value_counts().head(5))

# 从 mapping 反查 brand_id=0 对应的品牌名
print(f"\nbrand_id=0 对应的品牌名:")
# brand_to_id 是 {brand_name: id}, 反过来就是 {id: brand_name}
id_to_brand = {v: k for k, v in brand_to_id.items()}
print(f"   '{id_to_brand[0]}'")
print(f"   '{id_to_brand[1]}'")
print(f"   '{id_to_brand[2]}'")

brand_id=0 的商品数: 134,853
占比: 65.0%

brand_id 前 5 的分布:
brand_id
0     134853
7      13470
11      1027
21       774
19       748
Name: count, dtype: int64

brand_id=0 对应的品牌名:
   'Others'
   'L.A. COLORS'
   'NYX PROFESSIONAL MAKEUP'


In [10]:
# Cell 5: 保存
df_item_meta.to_csv(DATA_PROC / "item_meta_features.csv", index=False)
print(f"✅ 已保存 item_meta_features.csv")
print(f"   大小: {(DATA_PROC / 'item_meta_features.csv').stat().st_size / 1e6:.1f} MB")

# 保存映射
import json
with open(DATA_PROC / "sub_category_mapping.json", 'w') as f:
    json.dump({str(v): k for k, v in sub_cat_to_id.items()}, f, ensure_ascii=False, indent=2)

with open(DATA_PROC / "brand_mapping.json", 'w') as f:
    json.dump({str(v): k for k, v in brand_to_id.items()}, f, ensure_ascii=False, indent=2)

print(f"✅ 已保存 mapping 文件")

# 释放内存
del df_meta
gc.collect()
print(f"✅ 内存已释放")

✅ 已保存 item_meta_features.csv
   大小: 6.0 MB
✅ 已保存 mapping 文件


NameError: name 'df_meta' is not defined

In [11]:
%%time
# 重新加载所有要拼接的数据
# 注意: 你 Block 2-3 的内存里 df_item_meta 还在,但其他可能丢了

# 1. 主交易数据 (24.6M 行)
print("⏳ 加载主交易数据 (train_with_neg)...")
df_main = pd.read_csv(
    DATA_PROC / "train_with_neg.csv",
    dtype={'user_id': 'str', 'parent_asin': 'str', 'label': 'int8'},
)
print(f"   {len(df_main):,} 行")

# 2. 用户聚合特征 (Day 3)
user_feats = pd.read_csv(
    DATA_PROC / "user_activity.csv",
    dtype={
        'user_id': 'str',
        'user_interaction_count': 'int32',
        'user_avg_rating': 'float32',
        'user_last_timestamp': 'int64',
    },
)
print(f"   user_feats: {len(user_feats):,} 行")

# 3. 商品聚合特征 (Day 3)
item_feats = pd.read_csv(
    DATA_PROC / "item_popularity.csv",
    dtype={
        'parent_asin': 'str',
        'item_interaction_count': 'int32',
        'item_avg_rating': 'float32',
        'item_last_timestamp': 'int64',
    },
)
print(f"   item_feats: {len(item_feats):,} 行")

# 4. 新的 item meta 特征 (Block 3)
# df_item_meta 应该还在内存,如果不在就重新读
if 'df_item_meta' not in dir():
    df_item_meta = pd.read_csv(DATA_PROC / "item_meta_features.csv")
print(f"   df_item_meta: {len(df_item_meta):,} 行")

print(f"\n✅ 4 个数据源都准备好了")

⏳ 加载主交易数据 (train_with_neg)...
   24,653,142 行
   user_feats: 729,576 行
   item_feats: 207,385 行
   df_item_meta: 207,385 行

✅ 4 个数据源都准备好了
CPU times: user 10.3 s, sys: 425 ms, total: 10.8 s
Wall time: 10.9 s


In [12]:
%%time
# user_avg_price = 用户在 train 集买过的商品的平均价格
# 注意: 用原始 train (5.16M 行), 不用 train_with_neg (24.6M 含负样本)

# 重新读原始 train (我们需要的就是 user_id, parent_asin 两列)
df_train_raw = pd.read_csv(
    DATA_RAW / "BPC_5core_train.csv",
    usecols=['user_id', 'parent_asin'],
    dtype={'user_id': 'str', 'parent_asin': 'str'},
)
print(f"原始 train: {len(df_train_raw):,} 行")

# 拼上 item 价格 (用 Block 3 的 df_item_meta)
df_train_raw = df_train_raw.merge(
    df_item_meta[['parent_asin', 'price']],
    on='parent_asin', how='left',
)

# 按 user_id 算平均价 (注意: price 已经填过中位数,不会有 NaN)
user_avg_price = df_train_raw.groupby('user_id')['price'].mean().reset_index()
user_avg_price.columns = ['user_id', 'user_avg_price']
user_avg_price['user_avg_price'] = user_avg_price['user_avg_price'].astype('float32')

print(f"\nuser_avg_price 描述:")
print(user_avg_price['user_avg_price'].describe())

# 释放
del df_train_raw
gc.collect()
print(f"\n✅ 计算完成, {len(user_avg_price):,} 个用户")

原始 train: 5,165,289 行

user_avg_price 描述:
count    729576.000000
mean         20.022646
std          12.963072
min           1.690000
25%          14.075000
50%          16.970757
75%          21.813334
max        1158.660034
Name: user_avg_price, dtype: float64

✅ 计算完成, 729,576 个用户
CPU times: user 2.5 s, sys: 229 ms, total: 2.73 s
Wall time: 2.75 s


In [13]:
%%time
# LEFT JOIN 所有特征到主数据
# 顺序: user_feats → item_feats → item_meta → user_avg_price

print("⏳ 拼接特征...")

# 1. 用户聚合特征 (3 个)
df_main = df_main.merge(user_feats, on='user_id', how='left')
print(f"   ✅ 拼上 user_feats")

# 2. 商品聚合特征 (3 个)
df_main = df_main.merge(item_feats, on='parent_asin', how='left')
print(f"   ✅ 拼上 item_feats")

# 3. 商品 meta 特征 (6 个)
df_main = df_main.merge(df_item_meta, on='parent_asin', how='left')
print(f"   ✅ 拼上 item_meta")

# 4. 用户历史均价 (1 个)
df_main = df_main.merge(user_avg_price, on='user_id', how='left')
print(f"   ✅ 拼上 user_avg_price")

print(f"\n最终 shape: {df_main.shape}")
print(f"列: {list(df_main.columns)}")
print(f"内存: {df_main.memory_usage(deep=True).sum()/1e9:.2f} GB")

# 检查 NaN
print(f"\nNaN 统计 (理论上应该 0):")
print(df_main.isnull().sum()[df_main.isnull().sum() > 0])

# 释放
del user_feats, item_feats, df_item_meta, user_avg_price
gc.collect()

⏳ 拼接特征...
   ✅ 拼上 user_feats
   ✅ 拼上 item_feats
   ✅ 拼上 item_meta
   ✅ 拼上 user_avg_price

最终 shape: (24653142, 16)
列: ['user_id', 'parent_asin', 'label', 'user_interaction_count', 'user_avg_rating', 'user_last_timestamp', 'item_interaction_count', 'item_avg_rating', 'item_last_timestamp', 'price', 'price_missing', 'title_length', 'n_categories', 'sub_category_id', 'brand_id', 'user_avg_price']
内存: 3.43 GB

NaN 统计 (理论上应该 0):
Series([], dtype: int64)
CPU times: user 12 s, sys: 4.15 s, total: 16.1 s
Wall time: 17.6 s


0

In [14]:
%%time
# 现在 df_main 有 16 列,我们造 2 个交叉特征:

# 交叉特征 1: user_price_diff (价格匹配度)
# = abs(item_price - user_avg_price) / user_avg_price
# 表达"用户偏离平时价格段多远"
df_main['user_price_diff'] = (
    (df_main['price'] - df_main['user_avg_price']).abs() / 
    (df_main['user_avg_price'] + 1e-6)  # 避免除 0
).astype('float32')

# 交叉特征 2: pop_x_activity (热度活跃度乘积) - 直接呼应 Day 4 难点
# = log(user_interaction_count + 1) * log(item_interaction_count + 1)
df_main['pop_x_activity'] = (
    np.log1p(df_main['user_interaction_count']) * 
    np.log1p(df_main['item_interaction_count'])
).astype('float32')

print(f"✅ 交叉特征已造好")
print(f"\nuser_price_diff 描述:")
print(df_main['user_price_diff'].describe())
print(f"\npop_x_activity 描述:")
print(df_main['pop_x_activity'].describe())

print(f"\n最终 df_main: {df_main.shape}")
print(f"内存: {df_main.memory_usage(deep=True).sum()/1e9:.2f} GB")

✅ 交叉特征已造好

user_price_diff 描述:
count    2.465314e+07
mean     5.937757e-01
std      1.505273e+00
min      0.000000e+00
25%      1.536697e-01
50%      3.453933e-01
75%      6.022810e-01
max      4.154215e+02
Name: user_price_diff, dtype: float64

pop_x_activity 描述:
count    2.465314e+07
mean     8.753977e+00
std      5.533868e+00
min      9.609060e-01
25%      4.852175e+00
50%      7.300446e+00
75%      1.102636e+01
max      6.751028e+01
Name: pop_x_activity, dtype: float64

最终 df_main: (24653142, 18)
内存: 3.62 GB
CPU times: user 1.15 s, sys: 220 ms, total: 1.37 s
Wall time: 1.37 s


In [15]:
# 保存增强版训练数据 (因为造得很费时间, 万一 kernel 挂掉不用重来)
# 注意文件会很大(可能 2-3 GB), 但 gitignored 不用担心

print("⏳ 保存中间数据...")
df_main.to_parquet(DATA_PROC / "train_with_all_features.parquet", index=False)
print(f"✅ 已保存 train_with_all_features.parquet")
print(f"   大小: {(DATA_PROC / 'train_with_all_features.parquet').stat().st_size / 1e9:.2f} GB")

⏳ 保存中间数据...
✅ 已保存 train_with_all_features.parquet
   大小: 1.74 GB


In [16]:
%%time
from sklearn.model_selection import train_test_split

# 定义 15 个特征列
FEATURE_COLS_V2 = [
    # User 基础 (3)
    'user_interaction_count', 'user_avg_rating', 'user_last_timestamp',
    # Item 基础 (3)
    'item_interaction_count', 'item_avg_rating', 'item_last_timestamp',
    # Item meta (6)
    'price', 'price_missing', 'title_length', 'n_categories',
    'sub_category_id', 'brand_id',
    # 用户均价 (1)
    'user_avg_price',
    # 交叉 (2)
    'user_price_diff', 'pop_x_activity',
]

print(f"v2 特征数: {len(FEATURE_COLS_V2)}")
for i, col in enumerate(FEATURE_COLS_V2):
    print(f"  {i+1:2d}. {col}")

# 切分 (用一样的 random_state, 保证可比性)
train_idx, val_idx = train_test_split(
    df_main.index,
    test_size=0.2,
    random_state=42,
    stratify=df_main['label'],
)

X_train_v2 = df_main.loc[train_idx, FEATURE_COLS_V2]
y_train_v2 = df_main.loc[train_idx, 'label']
X_val_v2 = df_main.loc[val_idx, FEATURE_COLS_V2]
y_val_v2 = df_main.loc[val_idx, 'label']

print(f"\nX_train_v2: {X_train_v2.shape}")
print(f"X_val_v2:   {X_val_v2.shape}")
print(f"内存: X_train_v2 {X_train_v2.memory_usage(deep=True).sum()/1e9:.2f} GB")

v2 特征数: 15
   1. user_interaction_count
   2. user_avg_rating
   3. user_last_timestamp
   4. item_interaction_count
   5. item_avg_rating
   6. item_last_timestamp
   7. price
   8. price_missing
   9. title_length
  10. n_categories
  11. sub_category_id
  12. brand_id
  13. user_avg_price
  14. user_price_diff
  15. pop_x_activity

X_train_v2: (19722513, 15)
X_val_v2:   (4930629, 15)
内存: X_train_v2 1.97 GB
CPU times: user 9.53 s, sys: 1.27 s, total: 10.8 s
Wall time: 11.4 s


In [17]:
%%time
import lightgbm as lgb

# 关键: 告诉 LightGBM 哪些是 categorical features
CATEGORICAL_FEATURES = ['sub_category_id', 'brand_id', 'price_missing']

train_data_v2 = lgb.Dataset(
    X_train_v2, label=y_train_v2,
    categorical_feature=CATEGORICAL_FEATURES,
)
val_data_v2 = lgb.Dataset(
    X_val_v2, label=y_val_v2,
    reference=train_data_v2,
    categorical_feature=CATEGORICAL_FEATURES,
)

# 参数: 用 Day 4 调好的 (boost=500 + 早停)
params_v2 = {
    'objective': 'binary',
    'metric': ['binary_logloss', 'auc'],
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'num_threads': 8,
}

print("⏳ 开始训练 LightGBM v2...")
print(f"   特征数: {len(FEATURE_COLS_V2)} (vs v1 的 6)")
print(f"   Categorical 特征: {CATEGORICAL_FEATURES}")
print(f"   预期耗时: 15-30 分钟 (比 v1 慢, 因为特征更多)")
print()

model_v2 = lgb.train(
    params_v2,
    train_data_v2,
    num_boost_round=500,
    valid_sets=[train_data_v2, val_data_v2],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=20),
        lgb.log_evaluation(period=50),
    ],
)

print(f"\n✅ 训练完成")
print(f"   Best iteration: {model_v2.best_iteration}")
print(f"   Best val AUC: {model_v2.best_score['val']['auc']:.4f}")
print(f"   Best val LogLoss: {model_v2.best_score['val']['binary_logloss']:.4f}")
print(f"\n📊 对比:")
print(f"   v1 (6 特征,  500 轮): val AUC 0.7738")
print(f"   v2 ({len(FEATURE_COLS_V2)} 特征, {model_v2.best_iteration} 轮): val AUC {model_v2.best_score['val']['auc']:.4f}")
print(f"   提升: +{(model_v2.best_score['val']['auc'] - 0.7738):.4f}")

⏳ 开始训练 LightGBM v2...
   特征数: 15 (vs v1 的 6)
   Categorical 特征: ['sub_category_id', 'brand_id', 'price_missing']
   预期耗时: 15-30 分钟 (比 v1 慢, 因为特征更多)

Training until validation scores don't improve for 20 rounds
[50]	train's binary_logloss: 0.384606	train's auc: 0.77558	val's binary_logloss: 0.38468	val's auc: 0.77543
[100]	train's binary_logloss: 0.373743	train's auc: 0.789396	val's binary_logloss: 0.373906	val's auc: 0.789045
[150]	train's binary_logloss: 0.368704	train's auc: 0.796158	val's binary_logloss: 0.368948	val's auc: 0.795663
[200]	train's binary_logloss: 0.36538	train's auc: 0.800713	val's binary_logloss: 0.365709	val's auc: 0.800073
[250]	train's binary_logloss: 0.363206	train's auc: 0.803696	val's binary_logloss: 0.363597	val's auc: 0.802963
[300]	train's binary_logloss: 0.361335	train's auc: 0.806183	val's binary_logloss: 0.361797	val's auc: 0.805353
[350]	train's binary_logloss: 0.360141	train's auc: 0.807707	val's binary_logloss: 0.360667	val's auc: 0.806787
[400]	train

In [18]:
import json

# 保存模型
model_v2.save_model('../models/lightgbm_v2_meta_cross.txt')
print(f"✅ 模型已保存: lightgbm_v2_meta_cross.txt")

# v2 详细报告
v2_report = {
    'date': '2026-05-25',
    'day': 'Day 5 (项目部分)',
    'version': 'v2',
    'features': FEATURE_COLS_V2,
    'n_features': len(FEATURE_COLS_V2),
    'categorical_features': CATEGORICAL_FEATURES,
    'n_train': int(len(X_train_v2)),
    'n_val': int(len(X_val_v2)),
    'best_iteration': int(model_v2.best_iteration),
    'metrics': {
        'val_auc': float(model_v2.best_score['val']['auc']),
        'val_logloss': float(model_v2.best_score['val']['binary_logloss']),
        'train_auc': float(model_v2.best_score['train']['auc']),
        'train_logloss': float(model_v2.best_score['train']['binary_logloss']),
    },
    'comparison_with_previous': {
        'v0_day3_baseline': {'features': 6, 'val_auc': 0.7645, 'time_min': 2.73},
        'v1_day4_tuned':    {'features': 6, 'val_auc': 0.7738, 'time_min': 13.45},
        'v2_day5_meta':     {'features': 15, 'val_auc': float(model_v2.best_score['val']['auc']), 'time_min': 14.1},
        'absolute_improvement_v0_to_v2': round(float(model_v2.best_score['val']['auc']) - 0.7645, 4),
        'absolute_improvement_v1_to_v2': round(float(model_v2.best_score['val']['auc']) - 0.7738, 4),
    },
    'training_time_min': 14.1,
    'notes': 'Added 6 item meta features (price, category, brand, etc) + 2 cross features (price_diff, pop_x_activity). Confirmed Day 4 hypothesis: features > more boost rounds.',
}

with open('../models/v2_report.json', 'w') as f:
    json.dump(v2_report, f, indent=2, ensure_ascii=False)

print("✅ v2 报告已保存")

✅ 模型已保存: lightgbm_v2_meta_cross.txt
✅ v2 报告已保存


In [19]:
# 特征重要性 (gain) 排序
importance_gain = model_v2.feature_importance(importance_type='gain')
importance_split = model_v2.feature_importance(importance_type='split')

importance_df = pd.DataFrame({
    'feature': FEATURE_COLS_V2,
    'gain': importance_gain,
    'split': importance_split,
}).sort_values('gain', ascending=False).reset_index(drop=True)

print("=" * 70)
print("📊 v2 特征重要性 (Gain)")
print("=" * 70)
print(importance_df.to_string(index=False))

# 标记新特征 (Day 5 加的)
new_features = {
    'price', 'price_missing', 'title_length', 'n_categories',
    'sub_category_id', 'brand_id', 'user_avg_price',
    'user_price_diff', 'pop_x_activity',
}

print(f"\n📍 新特征在 Top 15 的排名:")
for rank, row in importance_df.iterrows():
    marker = "🆕 NEW" if row['feature'] in new_features else "    OLD"
    print(f"  {rank+1:2d}. {marker} {row['feature']:<30} gain={row['gain']:>12,.0f}")

📊 v2 特征重要性 (Gain)
               feature         gain  split
item_interaction_count 1.168383e+07   1304
   item_last_timestamp 4.466016e+06   1889
   user_last_timestamp 2.736802e+06   2125
       user_price_diff 2.124545e+06    814
        user_avg_price 1.870744e+06   1242
                 price 1.742186e+06    670
        pop_x_activity 1.430206e+06    412
              brand_id 1.418600e+06   3503
user_interaction_count 1.190998e+06   1412
          title_length 8.111182e+05    507
       sub_category_id 3.734505e+05    530
       item_avg_rating 2.467656e+05    326
       user_avg_rating 5.649259e+04    190
         price_missing 1.764104e+04     35
          n_categories 7.059626e+03     41

📍 新特征在 Top 15 的排名:
   1.     OLD item_interaction_count         gain=  11,683,826
   2.     OLD item_last_timestamp            gain=   4,466,016
   3.     OLD user_last_timestamp            gain=   2,736,802
   4. 🆕 NEW user_price_diff                gain=   2,124,545
   5. 🆕 NEW user_avg_pri

In [21]:
%%time
import numpy as np

# 在 val 上预测
df_main['y_pred_v2'] = np.nan
df_main.loc[val_idx, 'y_pred_v2'] = model_v2.predict(
    X_val_v2, num_iteration=model_v2.best_iteration
)

# 取出 val 集
df_val_v2 = df_main.loc[val_idx, ['user_id', 'parent_asin', 'label', 'y_pred_v2']].copy()
df_val_v2.rename(columns={'y_pred_v2': 'y_pred'}, inplace=True)

print(f"Val 集准备完成: {len(df_val_v2):,} 行")


def compute_user_metrics(group, k_list=[5, 10, 20]):
    sorted_group = group.sort_values('y_pred', ascending=False)
    labels = sorted_group['label'].values
    n_pos = labels.sum()
    n_total = len(labels)
    metrics = {}
    for k in k_list:
        k_actual = min(k, n_total)
        top_k_labels = labels[:k_actual]
        n_hits = top_k_labels.sum()
        metrics[f'precision@{k}'] = n_hits / k_actual if k_actual > 0 else 0
        metrics[f'recall@{k}'] = n_hits / n_pos if n_pos > 0 else np.nan
        gains = (2 ** top_k_labels - 1)
        discounts = 1 / np.log2(np.arange(k_actual) + 2)
        dcg = (gains * discounts).sum()
        n_pos_in_topk = min(n_pos, k_actual)
        ideal_labels = np.zeros(k_actual)
        ideal_labels[:int(n_pos_in_topk)] = 1
        ideal_gains = (2 ** ideal_labels - 1)
        idcg = (ideal_gains * discounts).sum()
        metrics[f'ndcg@{k}'] = dcg / idcg if idcg > 0 else np.nan
    return pd.Series(metrics)


print("⏳ 计算 v2 分用户 Top-K (预计 2-3 分钟)...")
user_metrics_v2 = df_val_v2.groupby('user_id').apply(
    compute_user_metrics, k_list=[5, 10, 20], include_groups=False,
)
summary_v2 = user_metrics_v2.mean()
print(f"✅ 计算完成")

Val 集准备完成: 4,930,629 行
⏳ 计算 v2 分用户 Top-K (预计 2-3 分钟)...
✅ 计算完成
CPU times: user 4min 22s, sys: 3.48 s, total: 4min 25s
Wall time: 2min 27s


In [22]:
v1_metrics = {
    'precision@5': 0.2053, 'precision@10': 0.1776, 'precision@20': 0.1694,
    'recall@5': 0.8800, 'recall@10': 0.9650, 'recall@20': 0.9906,
    'ndcg@5': 0.7333, 'ndcg@10': 0.7678, 'ndcg@20': 0.7777,
}

print("=" * 65)
print("分用户 Top-K v1 vs v2 对比")
print("=" * 65)
print(f"{'指标':<18}{'v1':<12}{'v2':<12}{'delta':<10}")
print("-" * 50)
for k in [5, 10, 20]:
    for metric in ['precision', 'recall', 'ndcg']:
        key = metric + '@' + str(k)
        v1 = v1_metrics[key]
        v2 = summary_v2[key]
        delta = v2 - v1
        sign = '+' if delta > 0 else ''
        print(f"{key:<18}{v1:<12.4f}{v2:<12.4f}{sign}{delta:.4f}")
    print()

分用户 Top-K v1 vs v2 对比
指标                v1          v2          delta     
--------------------------------------------------
precision@5       0.2053      0.2097      +0.0044
recall@5          0.8800      0.8941      +0.0141
ndcg@5            0.7333      0.7690      +0.0357

precision@10      0.1776      0.1787      +0.0011
recall@10         0.9650      0.9686      +0.0036
ndcg@10           0.7678      0.7990      +0.0312

precision@20      0.1694      0.1698      +0.0004
recall@20         0.9906      0.9916      +0.0010
ndcg@20           0.7777      0.8078      +0.0301

